# GEANT h96 reproducibility matrix — matrix2

This replacement notebook uses the separately versioned `matrix2` trainer namespace. Matrix lifecycle metadata is stored in a validated sibling sidecar, not inside the trainer-owned output directory. Preparation, tests, and dry-run do not train. The official test split is excluded.

Select a CPU Colab runtime, run the cells top-to-bottom, and upload exactly the four approved source/dataset ZIP and manifest files. Hashes, not upload filenames, determine identity. Review the generated review6 source candidate and audit before enabling the separately disabled training gate. The launcher asks for exact authorization before every run. Colab execution is not claimed until it is observed there.

In [ ]:
# Exact runtime requirements; this cell does not authorize training.
import sys, subprocess, importlib.metadata as ilm
if sys.version_info[:2] != (3, 12): raise RuntimeError(f'Requires Python 3.12; found {sys.version.split()[0]}')
PINS = {'numpy':'1.26.4','torch':'2.12.0','psutil':'7.2.2','pytest':'9.1.1'}
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*[f'{k}=={v}' for k,v in PINS.items()]],check=True)
actual = {k:ilm.version(k) for k in PINS}
if actual != PINS: raise RuntimeError(f'Pinned packages not active: {actual}; restart runtime and rerun from the top')
import torch
if torch.cuda.is_available(): raise RuntimeError('Select CPU runtime; CUDA is outside this protocol')
print('PASS: Python, pinned packages, CPU runtime', sys.version.split()[0], actual)

In [ ]:
# Hash-identify four uploaded files, verify bundles, and extract only to fresh session paths.
import hashlib, importlib, json, os, sys, uuid, zipfile
from pathlib import Path
from google.colab import files
EXPECTED = {
 'geant_h96_colab_source_v1.zip':'eec9dc3cadabc3a76c239ad8551f62995f294b9816a931ad71b8307c29082914',
 'source_snapshot_manifest.json':'458e25125697023967cc95be85cf45e5159d47f0797e4e3a73621f77d6eb70d2',
 'geant_h96_colab_trainval_v1.zip':'df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e',
 'dataset_bundle_manifest.json':'70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95',
}
def digest(payload): return hashlib.sha256(payload).hexdigest()
uploaded = files.upload()
if len(uploaded) != 4: raise RuntimeError(f'Upload exactly four approved files; received {len(uploaded)}')
by_hash = {}
for _name, body in uploaded.items(): by_hash.setdefault(digest(body), []).append(body)
resolved = {}
for name, expected_hash in EXPECTED.items():
    matches = by_hash.get(expected_hash, [])
    if len(matches) != 1: raise RuntimeError(f'Missing, duplicate, or mismatched approved file: {name}')
    resolved[name] = matches[0]
if sum(map(len, by_hash.values())) != 4: raise RuntimeError('Extra or duplicate upload refused')
SESSION_ID = uuid.uuid4().hex
SESSION = Path('/content') / f'geant_h96_matrix2_{SESSION_ID}'
SESSION.mkdir(exist_ok=False)
INCOMING = SESSION / 'incoming'; INCOMING.mkdir(exist_ok=False)
for name, body in resolved.items():
    with (INCOMING/name).open('xb') as stream: stream.write(body); stream.flush(); os.fsync(stream.fileno())
source_zip = INCOMING/'geant_h96_colab_source_v1.zip'; source_manifest = INCOMING/'source_snapshot_manifest.json'
data_zip = INCOMING/'geant_h96_colab_trainval_v1.zip'; data_manifest = INCOMING/'dataset_bundle_manifest.json'
sys.path.insert(0, str(source_zip))
pkg = importlib.import_module('phase9_geant.colab_h96_v1.package')
source_check = pkg.verify_source_bundle(source_zip, source_manifest, EXPECTED[source_zip.name], EXPECTED[source_manifest.name])
dataset_check = pkg.verify_dataset_bundle(data_zip, data_manifest, EXPECTED[data_zip.name], EXPECTED[data_manifest.name])
source_obj = json.loads(source_manifest.read_text(encoding='utf-8'))
REPO = SESSION/'approved_source_snapshot'
pkg.extract_source_bundle(source_zip, REPO, source_obj)
sys.path.remove(str(source_zip)); importlib.invalidate_caches(); sys.path.insert(0, str(REPO))
pkg.verify_extracted_source(REPO, source_obj)
pkg.extract_dataset_bundle(data_zip, REPO)
DATA_ROOT = REPO/pkg.DATA_ROOT_RELATIVE
if {p.name for p in DATA_ROOT.iterdir()} != set(pkg.DATA_ARTIFACTS): raise RuntimeError('Dataset exact allowlist mismatch')
metadata = pkg._read_dataset_metadata(DATA_ROOT)
if metadata['windows']['train_count'] != 3924 or metadata['windows']['validation_count'] != 953: raise RuntimeError('h96 window counts mismatch')
from phase9_geant.colab_h96_v1 import environment, linux_resources, matrix_workflow
ENV = environment.verify_environment(require_pytest=True)
if linux_resources.RESOURCE_POLICY_ID != 'colab-linux-process-memory-v3-visible-cgroup' or linux_resources.PROCESS_PEAK_LIMIT_BYTES != 2*1024**3: raise RuntimeError('Expected v3 2-GiB memory policy is not active')
print('PASS: exact source/data bundle hashes, member inventories, extraction, 9-file train/validation allowlist, topology metadata, and v3 policy')
print({'source_members':len(source_check['members']),'dataset_members':len(dataset_check['members']),'windows':metadata['windows'],'test_members':0,'policy':linux_resources.RESOURCE_POLICY_ID})

In [ ]:
# Complete notebook-configured synthetic suite and isolated matrix2 dry-run only.
import json, subprocess, sys, uuid
TESTS = ['phase9_geant/colab_h96_v1/tests/test_portability.py','phase9_geant/colab_h96_v1/tests/test_launch_matrix.py','phase9_geant/colab_h96_v1/tests/test_matrix_workflow.py','phase9_geant/tests/test_horizon96_streaming_experiment.py','phase9_geant/tests/test_horizon96_streaming_experiment_v2.py','phase9_geant/tests/test_spatiotemporal_link.py']
test_base = SESSION/f'pytest_{uuid.uuid4().hex}'
test = subprocess.run([sys.executable,'-m','pytest','-q','-rs','--basetemp',str(test_base),*TESTS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode != 0: raise RuntimeError(f'Synthetic suite failed with exit {test.returncode}')
ARGS = ['--repository-root',str(REPO),'--source-bundle',str(source_zip),'--source-manifest',str(source_manifest),'--expected-source-bundle-sha256',EXPECTED[source_zip.name],'--expected-source-manifest-sha256',EXPECTED[source_manifest.name],'--dataset-bundle',str(data_zip),'--dataset-manifest',str(data_manifest),'--expected-dataset-bundle-sha256',EXPECTED[data_zip.name],'--expected-dataset-manifest-sha256',EXPECTED[data_manifest.name]]
LAUNCH = REPO/'phase9_geant/colab_h96_v1/launch_matrix.py'
dry = subprocess.run([sys.executable,'-I','-S',str(LAUNCH),'--dry-run',*ARGS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(dry.stdout)
if dry.returncode != 0: raise RuntimeError(f'Matrix2 dry-run failed with exit {dry.returncode}')
DRY = json.loads(dry.stdout)
if DRY.get('status') != 'preflight_passed_no_training' or DRY.get('matrix_output_root') != 'geant_horizon96_streaming_experiment_v3_repro_matrix2': raise RuntimeError('Matrix2 dry-run identity mismatch')
expected_root = REPO/'data/processed/geant_horizon96_streaming_experiment_v3_repro_matrix2'
expected_sidecar = expected_root.parent/(expected_root.name+'.matrix_run.json')
if DRY.get('output_root') != str(expected_root) or DRY.get('matrix_manifest') != str(expected_sidecar): raise RuntimeError('Trainer namespace and matrix sidecar path mismatch')
if any(run['seed']==42 and run['adjacency_mode']=='topology' for run in DRY['planned_runs']): raise RuntimeError('Completed seed-42 topology must not be rerun')
if DRY.get('resource_policy',{}).get('policy_id') != 'colab-linux-process-memory-v3-visible-cgroup': raise RuntimeError('Resource policy mismatch')
PREFLIGHT = {'status':DRY['status'],'matrix_id':DRY['matrix_id'],'output_root':DRY['output_root'],'matrix_manifest':DRY['matrix_manifest'],'planned_runs':DRY['planned_runs'],'source_sha256':EXPECTED[source_zip.name],'dataset_sha256':EXPECTED[data_zip.name],'resource_policy':DRY['resource_policy'],'environment':ENV,'tests_output':test.stdout,'test_split_accessed':False}
(SESSION/'preflight.json').write_text(json.dumps(PREFLIGHT,indent=2),encoding='utf-8')
print('PASS: 6-file-group synthetic suite and matrix2 preflight; no training launched')

## Separate training authorization — disabled by default

Do not change these controls until you have reviewed and accepted the review5 audit. This runs the remaining eight experiments sequentially only; each run has its own exact authorization phrase. Existing matrix1 and seed-42 topology artifacts are not reused. A failed matrix2 namespace is preserved and cannot be resumed.

In [ ]:
SOURCE_REVIEW_APPROVED = False
RUN_MATRIX_AUTHORIZED = False
if SOURCE_REVIEW_APPROVED is not True: raise PermissionError('Blocked until review6 source audit is accepted')
if PREFLIGHT.get('status') != 'preflight_passed_no_training' or PREFLIGHT.get('test_split_accessed') is not False: raise RuntimeError('Current session has no valid matrix2 preflight receipt')
if input('Type AUTHORIZE H96 MATRIX2 to begin the eight remaining runs: ').strip() != 'AUTHORIZE H96 MATRIX2': raise PermissionError('No training authorization; nothing launched')
RUN_MATRIX_AUTHORIZED = True
command = [sys.executable,'-I','-S',str(LAUNCH),'--run-matrix',*ARGS]
log_path = SESSION/'matrix2_launcher.log'
with log_path.open('x',encoding='utf-8') as log:
    def emit(line): print(line,flush=True); log.write(line+'\n'); log.flush(); os.fsync(log.fileno())
    def authorize_one(phrase): return input('Type exactly '+phrase+':\n')
    exit_code = None
    try: exit_code = matrix_workflow.run_matrix_with_prompt_bridge(command,prompt_fn=authorize_one,output_fn=emit)
    finally:
        matrix_root = REPO/'data/processed/geant_horizon96_streaming_experiment_v3_repro_matrix2'
        matrix_record = matrix_root.parent/'geant_horizon96_streaming_experiment_v3_repro_matrix2.matrix_run.json'
        if matrix_record.is_file():
            export_path = SESSION/f'matrix2_export_{SESSION_ID}.zip'
            matrix_workflow.export_matrix_results(matrix_root,export_path,matrix_manifest_path=matrix_record)
            print('Matrix artifacts exported to',export_path)
            files.download(str(export_path))
    if exit_code != 0: raise RuntimeError(f'Matrix2 stopped with exit {exit_code}; preserve all artifacts and do not retry')
print('Matrix2 finished. Inspect its sibling matrix sidecar and all run manifests before interpreting results.')